In [ ]:
# Cell 1: Install arc-agi from competition wheels
import subprocess, sys
wheels = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--no-index', f'--find-links={wheels}',
    'arc-agi', 'python-dotenv'
])
print('[Cell1] arc-agi installed from competition wheels')


In [ ]:
# Cell 2: DenseExplorer BFS + VericodingAgent framework fallback
import os, hashlib, collections, random, json, time, gc, traceback
import numpy as np
import pandas as pd
from arcengine import GameAction
from agents.agent import Agent

_ALL_POSITIONS = [(x, y) for y in range(0, 64, 2) for x in range(0, 64, 2)]

def fast_grid_hash(grid):
    data = np.asarray(grid, dtype=np.int32).tobytes()
    return int(hashlib.md5(data).hexdigest()[:16], 16)

def safe_step(env, action, x=None, y=None):
    if hasattr(action, 'is_complex') and action.is_complex():
        if x is None or y is None:
            return None
        try:
            a = GameAction(int(action))
            a.set_data({'x': int(x), 'y': int(y)})
            return env.step(a)
        except (KeyError, TypeError, AttributeError):
            return None
    return env.step(action)

def is_win(frame):
    s = getattr(frame, "state", None)
    return s is not None and "WIN" in str(s)

def get_action_list(env):
    for attr in ("actions", "action_space"):
        try:
            raw = getattr(env, attr)
            acts = list(raw.actions if hasattr(raw, "actions") else raw)
            if acts and len(acts) > 0:
                return acts
        except Exception:
            continue
    try:
        acts = list(env.action_space)
        if acts:
            return acts
    except Exception:
        pass
    return []

class DenseExplorer:
    def __init__(self, env, action_list):
        self._env = env
        self._actions = action_list
        self._click_idx = next((i for i, a in enumerate(self._actions)
                                if hasattr(a, 'is_complex') and a.is_complex()), None)
        self._simple_indices = [i for i, a in enumerate(self._actions)
                                if not (hasattr(a, 'is_complex') and a.is_complex())]
        self._budget = 0
        self._total_steps = 0
        self.solution = None
        self.live_clicks = []

    def _safe_step(self, action_idx, cx=0, cy=0):
        self._total_steps += 1
        act = self._actions[action_idx]
        return safe_step(self._env, act, cx, cy)

    def _grid(self, frame_or_reset):
        fr = getattr(frame_or_reset, "frame", None)
        if fr is not None and len(fr) > 0:
            return np.asarray(fr[0], dtype=np.int32)
        return np.zeros((64, 64), dtype=np.int32)

    def _grid_any(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.astype(np.int32) if obj.ndim == 2 else obj[0].astype(np.int32)
        if isinstance(obj, tuple):
            return self._grid_any(obj[0])
        return self._grid(obj)

    def _phase1(self, budget):
        for aidx in self._simple_indices:
            if self._total_steps >= budget:
                return None
            self._env.reset()
            for _k in range(200):
                nf = self._safe_step(aidx)
                if nf is None:
                    break
                if is_win(nf):
                    return [aidx] * (_k + 1)
                if self._total_steps >= budget:
                    break
        return None

    def _phase2(self, max_pos=1024):
        live = []
        self._env.reset()
        if self._simple_indices:
            bf = self._safe_step(self._simple_indices[0])
        else:
            bf = self._safe_step(self._click_idx, 0, 0) if self._click_idx is not None else None
        base_hash = fast_grid_hash(self._grid(bf)) if bf else 0
        for pi, (px, py) in enumerate(_ALL_POSITIONS):
            if pi >= max_pos or self._total_steps >= self._budget:
                break
            self._env.reset()
            nf = self._safe_step(self._click_idx, px, py)
            if nf is None:
                continue
            if is_win(nf):
                self.solution = [(self._click_idx, px, py)]
                return "WIN", [(px, py, 0)]
            h = fast_grid_hash(self._grid(nf))
            if h != base_hash:
                live.append((px, py, h))
        return "LIVE" if live else "NONE", live

    def _phase3(self, live, max_steps):
        if not live:
            return False
        seen = set()
        frontier = []
        for px, py, h in live:
            if h not in seen:
                seen.add(h)
                frontier.append((h, [(self._click_idx, px, py)], [(px, py)]))
        explored = set(seen)
        while frontier and self._total_steps < max_steps:
            frontier.sort(key=lambda n: len(n[1]))
            cur_h, prefix, xy_list = frontier.pop(0)
            self._env.reset()
            ok = True
            for aidx, cx, cy in prefix:
                if self._safe_step(aidx, cx, cy) is None:
                    ok = False
                    break
            if not ok:
                continue
            for aidx in self._simple_indices:
                if self._total_steps >= max_steps:
                    return self.solution is not None
                nf = self._safe_step(aidx)
                if nf is None:
                    continue
                if is_win(nf):
                    self.solution = [a for a, _, _ in prefix] + [aidx]
                    return True
                nh = fast_grid_hash(self._grid(nf))
                if nh not in explored:
                    explored.add(nh)
                    frontier.append((nh, list(prefix) + [(aidx, 32, 32)], xy_list))
            for lcx, lcy in list(dict.fromkeys([(x, y) for x, y, _ in live]))[:10]:
                if self._click_idx is not None:
                    if self._total_steps >= max_steps:
                        break
                    nf = self._safe_step(self._click_idx, lcx, lcy)
                    if nf is None:
                        continue
                    if is_win(nf):
                        self.solution = [a for a, _, _ in prefix] + [(self._click_idx, lcx, lcy)]
                        return True
                    nh = fast_grid_hash(self._grid(nf))
                    if nh not in explored:
                        explored.add(nh)
                        frontier.append((nh, list(prefix) + [(self._click_idx, lcx, lcy)], [(lcx, lcy)]))
        return self.solution is not None

    def explore(self, max_steps=2000):
        self._budget = max_steps
        self._total_steps = 0
        self.solution = None
        self.live_clicks = []
        if self._simple_indices:
            sol = self._phase1(max_steps)
            if sol:
                self.solution = sol
                return True
        if self._click_idx is not None and self._total_steps < self._budget:
            result, data = self._phase2(1024)
            if result == "WIN":
                return True
            if result == "LIVE":
                self.live_clicks = data
        if self.live_clicks and self._total_steps < self._budget:
            self._phase3(self.live_clicks, max_steps // 2)
        return self.solution is not None

class VericodingAgent(Agent):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._step = 0
        self._graph = collections.defaultdict(dict)
        self._visited = set()
        self._last_hash = None
        self._last_key = None
        rng = np.random.RandomState(42)
        self._zob = rng.randint(0, 2**63-1, size=(64, 64, 16), dtype=np.uint64)
    def _hash(self, grid):
        best = np.uint64(2**64-1)
        for rot in range(4):
            for flip in (False, True):
                g = np.rot90(grid, rot)
                if flip: g = np.fliplr(g)
                gh, gw = g.shape[:2]
                onehot = np.eye(16, dtype=np.uint64)[np.clip(g, 0, 15)]
                hv = int(np.bitwise_xor.reduce(self._zob[:gh, :gw, :] * onehot))
                if hv < int(best): best = np.uint64(hv)
        return int(best)
    def _get_action_idx(self, action):
        if isinstance(action, int): return action
        if hasattr(action, 'action_type'): return action.action_type
        return int(action)
    def _make_action(self, idx, grid, frame):
        a = GameAction(int(idx))
        if hasattr(a, 'is_complex') and a.is_complex():
            y = random.randint(0, max(0, grid.shape[0] - 1))
            x = random.randint(0, max(0, grid.shape[1] - 1))
            a.set_data({'x': int(x), 'y': int(y)})
            return a, (int(idx), int(x), int(y))
        return a, int(idx)
    def choose_action(self, frames, latest_frame) -> GameAction:
        try: return self._fsm(frames, latest_frame)
        except Exception: return self._fallback(latest_frame)
    def _fsm(self, frames, latest_frame) -> GameAction:
        self._step += 1
        grid = np.array(latest_frame.frame[0], dtype=np.int32)
        raw = latest_frame.available_actions if latest_frame.available_actions else []
        if not raw: return GameAction(1)
        h, w = grid.shape[:2]
        cur_hash = self._hash(grid)
        self._visited.add(cur_hash)
        if self._last_hash is not None and self._last_key is not None:
            self._graph[self._last_hash][self._last_key] = cur_hash
        self._last_hash = cur_hash
        complex_a, simple_a = [], []
        for a in raw:
            idx = self._get_action_idx(a)
            if idx in (6, 7, 8) or (hasattr(a, 'is_complex') and a.is_complex()):
                complex_a.append(idx)
            else:
                simple_a.append(idx)
        nz = np.count_nonzero(grid)
        if complex_a and nz > 0:
            flat = grid.flatten()
            nzi = np.where(flat > 0)[0]
            pi = nzi[self._step % len(nzi)]
            y, x = divmod(int(pi), w)
            a = GameAction(complex_a[0])
            a.set_data({'x': int(x), 'y': int(y)})
            self._last_key = (complex_a[0], int(x), int(y))
            return a
        if simple_a:
            ci = int(simple_a[self._step % len(simple_a)])
            a, k = self._make_action(ci, grid, latest_frame)
            self._last_key = k
            return a
        return self._fallback(latest_frame)
    def _fallback(self, frame) -> GameAction:
        raw = getattr(frame, 'available_actions', None) or []
        if not raw: return GameAction(1)
        c = raw[self._step % len(raw)]
        a, _ = self._make_action(self._get_action_idx(c), getattr(frame, 'frame', [[0]]), frame)
        return a

# ── Write framework fallback agent ──
with open("/tmp/my_agent.py", "w") as f:
    f.write("""
import random, collections
import numpy as np
from arcengine import GameAction
from agents.agent import Agent

class VericodingAgent(Agent):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._step = 0
        self._graph = collections.defaultdict(dict)
        self._visited = set()
        self._last_hash = None
        self._last_key = None
        rng = np.random.RandomState(42)
        self._zob = rng.randint(0, 2**63-1, size=(64, 64, 16), dtype=np.uint64)
    def _hash(self, grid):
        best = np.uint64(2**64-1)
        for rot in range(4):
            for flip in (False, True):
                g = np.rot90(grid, rot)
                if flip: g = np.fliplr(g)
                gh, gw = g.shape[:2]
                onehot = np.eye(16, dtype=np.uint64)[np.clip(g, 0, 15)]
                hv = int(np.bitwise_xor.reduce(self._zob[:gh, :gw, :] * onehot))
                if hv < int(best): best = np.uint64(hv)
        return int(best)
    def _get_action_idx(self, action):
        if isinstance(action, int): return action
        if hasattr(action, 'action_type'): return action.action_type
        return int(action)
    def _make_action(self, idx, grid, frame):
        a = GameAction(int(idx))
        if hasattr(a, 'is_complex') and a.is_complex():
            y = random.randint(0, max(0, grid.shape[0] - 1))
            x = random.randint(0, max(0, grid.shape[1] - 1))
            a.set_data({'x': int(x), 'y': int(y)})
            return a, (int(idx), int(x), int(y))
        return a, int(idx)
    def choose_action(self, frames, latest_frame) -> GameAction:
        try: return self._fsm(frames, latest_frame)
        except Exception: return self._fallback(latest_frame)
    def _fsm(self, frames, latest_frame) -> GameAction:
        self._step += 1
        grid = np.array(latest_frame.frame[0], dtype=np.int32)
        raw = latest_frame.available_actions if latest_frame.available_actions else []
        if not raw: return GameAction(1)
        h, w = grid.shape[:2]
        cur_hash = self._hash(grid)
        self._visited.add(cur_hash)
        if self._last_hash is not None and self._last_key is not None:
            self._graph[self._last_hash][self._last_key] = cur_hash
        self._last_hash = cur_hash
        complex_a, simple_a = [], []
        for a in raw:
            idx = self._get_action_idx(a)
            if idx in (6, 7, 8) or (hasattr(a, 'is_complex') and a.is_complex()):
                complex_a.append(idx)
            else:
                simple_a.append(idx)
        nz = np.count_nonzero(grid)
        if complex_a and nz > 0:
            flat = grid.flatten()
            nzi = np.where(flat > 0)[0]
            pi = nzi[self._step % len(nzi)]
            y, x = divmod(int(pi), w)
            a = GameAction(complex_a[0])
            a.set_data({'x': int(x), 'y': int(y)})
            self._last_key = (complex_a[0], int(x), int(y))
            return a
        if simple_a:
            ci = int(simple_a[self._step % len(simple_a)])
            a, k = self._make_action(ci, grid, latest_frame)
            self._last_key = k
            return a
        return self._fallback(latest_frame)
    def _fallback(self, frame) -> GameAction:
        raw = getattr(frame, 'available_actions', None) or []
        if not raw: return GameAction(1)
        c = raw[self._step % len(raw)]
        a, _ = self._make_action(self._get_action_idx(c), getattr(frame, 'frame', [[0]]), frame)
        return a
""")
print('[Cell2] V25.1 agent written to /tmp/my_agent.py')


In [ ]:
# Cell 3: Phase B — Competition Rerun (direct DenseExplorer + framework fallback)
import os, subprocess, shutil, sys

if os.environ.get("KAGGLE_IS_COMPETITION_RERUN"):
    print("[Cell3] Phase B: competition rerun detected")

    # Wait for gateway sidecar
    subprocess.run([
        "curl", "--fail", "--retry", "30",
        "--retry-delay", "2", "--max-time", "5",
        "http://gateway:8001/api/games"
    ])

    # Strategy A: Direct DenseExplorer on each game
    print("[Cell3] Strategy A: DenseExplorer BFS on each game...")
    try:
        from arc_agi import Arcade
        arcade = Arcade()
        # Try get_environments() (0.9.9) or available_environments (0.9.7)
        if hasattr(arcade, 'get_environments'):
            games = arcade.get_environments()
        elif hasattr(Arcade, 'available_environments'):
            games = Arcade.available_environments
        elif hasattr(arcade, 'available_environments'):
            games = arcade.available_environments
        else:
            games = []

        results = {}
        for g in games:
            gid = g.game_id if hasattr(g, 'game_id') else str(g)
            print(f"  [{gid}] starting...")
            try:
                env = arcade.make(gid)
                al = get_action_list(env)
                if not al:
                    print(f"  [{gid}] empty action list")
                    results[gid] = 0.0
                    env.close()
                    continue

                explorer = DenseExplorer(env, al)
                found = explorer.explore(max_steps=5000)
                if found and explorer.solution:
                    env.reset()
                    score = 0.0
                    for item in explorer.solution:
                        if isinstance(item, (list, tuple)) and len(item) == 3:
                            aidx, cx, cy = int(item[0]), int(item[1]), int(item[2])
                        else:
                            aidx, cx, cy = int(item), 32, 32
                        if aidx >= len(al): break
                        f = safe_step(env, al[aidx], cx, cy)
                        if f is None: break
                        score += float(getattr(f, "reward", 0.0))
                        if getattr(f, "done", False): break
                    results[gid] = score
                    print(f"  [{gid}] dense WIN sc={score:.4f}")
                else:
                    results[gid] = 0.0
                    print(f"  [{gid}] dense no solution")
                env.close()
            except Exception as e:
                print(f"  [{gid}] ERROR: {e}")
                traceback.print_exc()
                results[gid] = 0.0

        solved = sum(1 for v in results.values() if v > 0.0)
        mean = sum(results.values()) / max(1, len(results))
        print(f"[Cell3] DenseExplorer: {solved}/{len(results)} solved mean={mean:.4f}")

        # Write submission.parquet
        rows = []
        for gid, score in results.items():
            rows.append({'row_id': f'{gid}_0', 'game_id': str(gid), 'end_of_game': True, 'score': score})
        pd.DataFrame(rows).to_parquet('/kaggle/working/submission.parquet', index=False)
        print(f"[Cell3] submission.parquet written ({len(rows)} games)")
    except Exception as e:
        print(f"[Cell3] Strategy A failed: {e}")
        traceback.print_exc()

        # Strategy B: Framework fallback
        print("[Cell3] Strategy B: framework main.py...")
        src = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/agents"
        if os.path.isdir(src):
            shutil.copytree(src, "/kaggle/working/agents", dirs_exist_ok=True)
            print("[Cell3] framework copied")
        else:
            print(f"[Cell3] WARNING: {src} not found")
        shutil.copy("/tmp/my_agent.py", "/kaggle/working/agents/my_agent.py")
        init_code = "from agents.my_agent import VericodingAgent\nAVAILABLE_AGENTS = {'vericoding': VericodingAgent}"
        with open("/kaggle/working/agents/__init__.py", "w") as f:
            f.write(init_code)
        os.chdir("/kaggle/working")
        subprocess.run([sys.executable, "main.py", "--agent", "vericoding", "--output", "/kaggle/working/submission.parquet"])
        print("[Cell3] Strategy B complete")
else:
    print("[Cell3] Phase A: skipping (no competition rerun)")


In [ ]:
# Cell 4: Phase A — Write dummy submission.parquet (required to enable Submit button)
import os
import pandas as pd

if not os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    print('[Cell4] Phase A: writing dummy submission.parquet')
    pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score']
    ).to_parquet('/kaggle/working/submission.parquet', index=False)
    print('[Cell4] submission.parquet written — click Submit to Competition on Kaggle')
else:
    print('[Cell4] Phase B: submission.parquet handled by Cell 3')
